In [0]:
# 1. Validación del entorno

import sys
import pyspark
from delta import __version__ as delta_version

print(f"Python:     {sys.version.split()[0]}")
print(f"PySpark:    {pyspark.__version__}")
print(f"Delta Lake: {delta_version}")
print(f"Spark:      {spark.version}")
print(f"Usuario:    {spark.sql('SELECT current_user()').collect()[0][0]}")

Python:     3.12.3
PySpark:    4.1.0
Delta Lake: 3.4.0
Spark:      4.1.0
Usuario:    jjmoraleg1@eafit.edu.co


In [0]:
# 2. Variables del proyecto

CATALOG = "workspace"
SCHEMA  = "si7006_t2"

VOL_RAW         = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
VOL_STREAM_IN   = f"/Volumes/{CATALOG}/{SCHEMA}/stream_input"
VOL_CHECKPOINTS = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

TBL_BRONZE      = f"{CATALOG}.{SCHEMA}.orders_bronze"
TBL_SILVER      = f"{CATALOG}.{SCHEMA}.orders_silver"
TBL_GOLD_KPI    = f"{CATALOG}.{SCHEMA}.gold_kpi_ventana"
TBL_GOLD_ALERTS = f"{CATALOG}.{SCHEMA}.gold_alertas_reorder"

print(f"Catálogo:     {CATALOG}")
print(f"Esquema:      {SCHEMA}")
print(f"Raw data:     {VOL_RAW}")
print(f"Stream input: {VOL_STREAM_IN}")
print(f"Checkpoints:  {VOL_CHECKPOINTS}")
print(f"Tabla Bronze: {TBL_BRONZE}")
print(f"Tabla Silver: {TBL_SILVER}")

Catálogo:     workspace
Esquema:      si7006_t2
Raw data:     /Volumes/workspace/si7006_t2/raw_data
Stream input: /Volumes/workspace/si7006_t2/stream_input
Checkpoints:  /Volumes/workspace/si7006_t2/checkpoints
Tabla Bronze: workspace.si7006_t2.orders_bronze
Tabla Silver: workspace.si7006_t2.orders_silver


In [0]:
# 3. Verificar que el CSV existe en raw_data

csv_path = f"{VOL_RAW}/OnlineRetail.csv"

for f in dbutils.fs.ls(VOL_RAW):
    print(f"{f.name}  ({f.size:,} bytes)")

df_muestra = (
    spark.read
        .option("header", "true")
        .option("encoding", "ISO-8859-1")
        .csv(csv_path)
)

print(f"\nFilas totales: {df_muestra.count():,}")
df_muestra.printSchema()
df_muestra.show(5, truncate=False)

OnlineRetail.csv  (45,580,638 bytes)

Filas totales: 541,909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84

In [0]:
# 4. Crear el schema en Unity Catalog

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
|si7006_t2         |
+------------------+



In [0]:
# 5. Para resetear antes de la demostracion

spark.sql(f"DROP TABLE IF EXISTS {TBL_BRONZE}")
spark.sql(f"DROP TABLE IF EXISTS {TBL_SILVER}")
dbutils.fs.rm(VOL_CHECKPOINTS, recurse=True)
dbutils.fs.mkdirs(VOL_CHECKPOINTS)
for f in dbutils.fs.ls(VOL_STREAM_IN):
    dbutils.fs.rm(f.path)
print("Reset completo.")

Reset completo.
